# PMT waveform cleaning: FFT, Savitzky–Golay, and their combination

This notebook compares the FFT brick-wall low-pass filter used in `PMT_PROCESSING.ipynb`, Savitzky–Golay smoothing, and a sequential FFT-plus-Savitzky–Golay filter using the 08_09 normal negative-trigger dark data. The purpose is not to choose the prettiest trace. It is to select a method that:

1. does not turn measured pre-trigger noise into a pulse;
2. retains injected PMT-like signals at a fixed false-positive probability;
3. preserves pulse height, charge, and timing;
4. remains acceptable over a neighborhood of filter settings.

All thresholds are learned from training noise and evaluated on files that were not used to construct the pulse template or threshold. Results remain embedded in this notebook; it writes no CSV or PNG files.


In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import SRC.KingCRAB.pmt_filter as pmt_filter_helpers
from SRC.KingCRAB.context import configure_module
from SRC.KingCRAB.pmt_filter import apply_filter, baseline_subtract, config_name, load_files, paired_detection_matrix

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.signal import savgol_filter
from scipy.stats import beta
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
ROOT=PROJECT_ROOT
if ROOT.name=='CODE':ROOT=ROOT.parent
DATA_FOLDER=Path('/Volumes/Untitled/08_09_Background_Normal')
if not DATA_FOLDER.exists():raise FileNotFoundError(f'Mount the data drive at {DATA_FOLDER}')

# File-disjoint development and validation samples. The ordering is fixed and
# declared before any performance result is calculated.
TRAIN_FILES=tuple(range(0,10))
TEST_FILES=tuple(range(50,70))
BASELINE_RANGE_NS=(-100.0,-80.0)
NULL_SEARCH_RANGE_NS=(-75.0,-20.0)
PULSE_RANGE_NS=(-20.0,40.0)
INJECTION_TIME_NS=-47.5
TARGET_FALSE_POSITIVE_RATES=(1e-2,3e-3,1e-3)
AMPLITUDE_SCALES=(0.5,0.75,1.0,1.5,2.0)
R_LOAD_OHM=50.0
RNG_SEED=8092026

## 1. Load real normal-trigger records

Each LeCroy file contains 500 segments. Training files provide the pulse template and empirical decision thresholds. Test files provide the final null and injection measurements. Splitting by file prevents neighboring waveform segments from appearing on both sides of the validation.


In [ ]:
configure_module(pmt_filter_helpers, globals())

t,train_raw=load_files(TRAIN_FILES)
t_test,test_raw=load_files(TEST_FILES)
if len(t)!=len(t_test):raise ValueError('Training and test records have different sample counts')
dt_train=np.median(np.diff(t));dt_test=np.median(np.diff(t_test))
if not np.isclose(dt_train,dt_test,rtol=1e-5):raise ValueError('Training and test sampling intervals differ')
# LeCroy exports can shift the nominal grid by a fraction of a sample between
# files. Express both on the training trigger-relative grid; voltages are not
# interpolated because their spacing and sample count are identical.
grid_offset_ns=float(np.median(t_test-t))
print(f'Test-file nominal time-grid offset: {grid_offset_ns:.4f} ns')
dt_s=np.median(np.diff(t))*1e-9;sample_rate_Hz=1/dt_s
base=(t>=BASELINE_RANGE_NS[0])&(t<BASELINE_RANGE_NS[1])
null_region=(t>=NULL_SEARCH_RANGE_NS[0])&(t<=NULL_SEARCH_RANGE_NS[1])
pulse_region=(t>=PULSE_RANGE_NS[0])&(t<=PULSE_RANGE_NS[1])

train=baseline_subtract(train_raw);test=baseline_subtract(test_raw)
print(f'Training records: {len(train):,}; held-out records: {len(test):,}')
print(f'{len(t)} samples/record, {dt_s*1e9:.3f} ns/sample, {sample_rate_Hz/1e9:.3f} GS/s')

## 2. Learn a PMT-like injection from training data only

The negative-trigger data contain a synchronized pulse near zero. A template is formed from **every training waveform** after aligning its minimum in the pulse region. The template is scaled to the median charge of the complete training population. No pulse-height preselection is used. This makes the tested signal morphology representative of the full normal-trigger dark table rather than only its large-pulse tail.

For validation, only the pre-trigger part of each held-out waveform is used. It predates the hardware-trigger pulse and therefore supplies real electronic-plus-dark baseline noise that was not selected for a fluctuation at the injection time.


In [ ]:
template_population=train
target_index=np.flatnonzero(pulse_region)[np.argmin(template_population[:,pulse_region],axis=1)]
center_index=np.flatnonzero(pulse_region)[len(np.flatnonzero(pulse_region))//2]
aligned=[]
for row,k in zip(template_population,target_index):aligned.append(np.roll(row,center_index-k))
template_full=np.mean(aligned,axis=0)
template_full-=template_full[base].mean()

# Retain only the local impulse, avoiding unrelated portions of the average.
local=(t>=-15)&(t<=25)
template=np.zeros_like(t);template[local]=template_full[local]
template_charge=trapezoid(-template[local],t[local]*1e-9)/R_LOAD_OHM
training_charge=[]
for row in template_population:training_charge.append(trapezoid(-row[pulse_region],t[pulse_region]*1e-9)/R_LOAD_OHM)
target_charge=np.median(np.asarray(training_charge))
if template_charge>0:template*=target_charge/template_charge

shift_samples=int(round((INJECTION_TIME_NS-t[center_index])/np.median(np.diff(t))))
injection_template=np.roll(template,shift_samples)
inject_region=(t>=INJECTION_TIME_NS-20)&(t<=INJECTION_TIME_NS+25)

fig,ax=plt.subplots(1,2,figsize=(13,4.2))
ax[0].plot(t,template_full*1e3,label='aligned all-waveform mean',alpha=.65)
ax[0].plot(t,template*1e3,label='local injection template',lw=2)
ax[0].set(xlim=(-30,50),xlabel='time [ns]',ylabel='voltage [mV]',title='Template learned from training files');ax[0].legend()
for row in test[:200]:ax[1].plot(t,row*1e3,color='0.5',alpha=.025)
ax[1].plot(t,injection_template*1e3,color='tab:red',lw=2,label='1× injected signal')
ax[1].set(xlim=(-100,-15),xlabel='time [ns]',ylabel='voltage [mV]',title='Held-out pre-trigger noise and injection');ax[1].legend()
fig.tight_layout();plt.show()
print(f'Template charge: {target_charge*1e12:.4f} pC; peak: {-template.min()*1e3:.3f} mV')

## 3. Predeclared filter families

The FFT implementation is the same brick-wall operation as `PMT_PROCESSING.ipynb`. The Savitzky–Golay grid spans several physical window widths and polynomial orders. The combined family first applies the FFT cutoff and then applies a local second-order Savitzky–Golay smoother. Because both operations are linear in the waveform interior, the combination tests whether removing broad high-frequency content before local polynomial smoothing provides information beyond either method alone. The unfiltered waveform is retained as the control. No single setting is selected before the held-out comparison.


In [ ]:
FFT_CUTOFF_MHZ=(40,60,80,100,150,250,400)
SG_WINDOW_NS=(1.0,1.8,3.0,5.0,8.0)
SG_POLYORDER=(2,3)
configs=[('raw',None,None)]
configs += [('fft',f*1e6,None) for f in FFT_CUTOFF_MHZ]
for width in SG_WINDOW_NS:
    samples=max(5,int(round(width/(dt_s*1e9))))
    if samples%2==0:samples+=1
    for order in SG_POLYORDER:
        if samples>order:configs.append(('savgol',samples,order))
# Cross representative mild, intermediate, and strong local windows with the
# complete FFT cutoff grid. Polynomial order two is used for this family so the
# combination scan tests complementary smoothing rather than duplicating both
# near-identical order-2/order-3 five-sample cases.
for cutoff_mhz in FFT_CUTOFF_MHZ:
    for samples in (5,9,15):configs.append(('combined',cutoff_mhz*1e6,(samples,2)))

configure_module(pmt_filter_helpers, globals())


print(f'Comparing {len(configs)} predeclared configurations.')

## 4. Held-out false positives and injected-signal recovery

For each filter, a detection threshold is the empirical training-noise quantile corresponding to a stated false-positive target. It is then frozen. The reported false-positive rate and detection efficiency come from held-out files. Thus smoothing cannot win merely because it lowers the RMS; its filtered noise distribution sets its own threshold.

In this notebook, **efficiency** means signal-injection detection efficiency. For filter configuration $f$, injected amplitude $a$, and threshold associated with target false-positive probability $\alpha$,

\[
\epsilon_{f}(a,\alpha)=\frac{\text{held-out injected records passing the frozen threshold}}{\text{held-out records injected at amplitude }a}.
\]

It is a conditional reconstruction probability: the waveform is already known to contain the injected template. It is not the R7378A quantum efficiency, the 128 nm photon-detection efficiency, the probability that a cosmogenic muon produces light, or the final experimental trigger efficiency. `mean_efficiency` is the unweighted mean over the five declared injection amplitudes and three false-positive targets. `worst_efficiency` is the smallest efficiency among those 15 operating points. The complete efficiency-versus-amplitude curves should be used when a particular physical signal scale is known; the mean is only a compact filter-comparison summary.

Charge is integrated over the known injection interval only for validation. Timing is the position of the negative minimum. The filter does not receive the true time during threshold construction.


In [ ]:
rng=np.random.default_rng(RNG_SEED)
# Randomly pair injection amplitude with records while keeping every method paired.
amplitudes=np.resize(np.asarray(AMPLITUDE_SCALES),len(test));rng.shuffle(amplitudes)
injected=test+amplitudes[:,None]*injection_template
rows=[];shape_rows=[]

for kind,p1,p2 in configs:
    name=config_name(kind,p1,p2)
    train_f=apply_filter(train,kind,p1,p2);test_f=apply_filter(test,kind,p1,p2)
    injected_f=apply_filter(injected,kind,p1,p2)
    train_score=-train_f[:,null_region].min(axis=1)
    test_score=-test_f[:,null_region].min(axis=1)
    signal_score=-injected_f[:,null_region].min(axis=1)

    clean_template=apply_filter(injection_template[None,:],kind,p1,p2)[0]
    peak_ratio=(-clean_template.min())/(-injection_template.min())
    charge_raw=trapezoid(-injection_template[inject_region],t[inject_region]*1e-9)/R_LOAD_OHM
    charge_filtered=trapezoid(-clean_template[inject_region],t[inject_region]*1e-9)/R_LOAD_OHM
    shape_rows.append({'configuration':name,'family':kind,'peak_ratio':peak_ratio,
                       'charge_ratio':charge_filtered/charge_raw})

    minima=np.flatnonzero(null_region)[np.argmin(injected_f[:,null_region],axis=1)]
    timing_error=np.abs(t[minima]-INJECTION_TIME_NS)
    for target_fpr in TARGET_FALSE_POSITIVE_RATES:
        threshold=np.quantile(train_score,1-target_fpr,method='higher')
        false_positive=test_score>threshold
        k=int(false_positive.sum());n=len(false_positive)
        fpr_upper=beta.ppf(.95,k+0.5,n-k+0.5)
        for amp in AMPLITUDE_SCALES:
            selected=amplitudes==amp
            detected=signal_score[selected]>threshold
            rows.append({'configuration':name,'family':kind,'target_fpr':target_fpr,
                         'threshold_mV':threshold*1e3,'test_false_positives':k,
                         'test_fpr':false_positive.mean(),'test_fpr_upper95':fpr_upper,
                         'amplitude':amp,'efficiency':detected.mean(),
                         'median_abs_timing_error_ns':np.median(timing_error[selected])})

results=pd.DataFrame(rows);shape=pd.DataFrame(shape_rows)
summary=(results.groupby(['configuration','family']).agg(
    mean_efficiency=('efficiency','mean'),worst_efficiency=('efficiency','min'),
    max_test_fpr=('test_fpr','max'),max_fpr_upper95=('test_fpr_upper95','max'),
    median_timing_error_ns=('median_abs_timing_error_ns','median')).reset_index())
summary=summary.merge(shape,on=['configuration','family'])
summary['peak_bias']=abs(summary.peak_ratio-1)
summary['charge_bias']=abs(summary.charge_ratio-1)
display(summary.sort_values('mean_efficiency',ascending=False).round(5))

## 5. Parameter-robust decision

There is no unique, assumption-free scalar score for denoising. To prevent a private preference from deciding the answer, the notebook reports the best configuration under three independently stated signal-distortion tolerances: 5%, 10%, and 20%. Within each tolerance, the selection rule is simply the largest mean held-out detection efficiency across all injected amplitudes and false-positive targets. The empirical false-positive result remains visible rather than being hidden inside a weighted score.

A method is recommended only if the same family remains competitive across these tolerances and neighboring settings. Otherwise the scientifically honest result is that filtering is not justified by this data set.


In [ ]:
choices=[]
for tolerance in (.05,.10,.20):
    eligible=summary[(summary.peak_bias<=tolerance)&(summary.charge_bias<=tolerance)]
    if len(eligible):
        best=eligible.sort_values(['mean_efficiency','max_fpr_upper95'],ascending=[False,True]).iloc[0]
        choices.append({'allowed_distortion':tolerance,'configuration':best.configuration,
                        'family':best.family,'mean_efficiency':best.mean_efficiency,
                        'worst_efficiency':best.worst_efficiency,'max_test_fpr':best.max_test_fpr,
                        'peak_bias':best.peak_bias,'charge_bias':best.charge_bias})
choices=pd.DataFrame(choices);display(choices.round(5))

# Paired record bootstrap against the raw control. Resampling records (rather
# than treating the three thresholds as independent events) preserves their
# correlation. This is a descriptive post-selection interval, so a marginal
# advantage is not treated as proof that filtering is required.
winner_name=choices.configuration.mode().iloc[0] if len(choices) else 'raw'
configure_module(pmt_filter_helpers, globals())

candidate_detect=paired_detection_matrix(winner_name)
raw_detect=paired_detection_matrix('raw')
paired_delta=candidate_detect.astype(float)-raw_detect.astype(float)
bootstrap_rng=np.random.default_rng(RNG_SEED+1);bootstrap_delta=[]
for _ in range(2000):
    index=bootstrap_rng.integers(len(test),size=len(test))
    bootstrap_delta.append(paired_delta[:,index].mean())
delta_mean=float(paired_delta.mean())
delta_ci=np.quantile(bootstrap_delta,[.025,.975])
print(f'{winner_name} minus raw mean efficiency: {delta_mean:+.5f} '
      f'(paired record-bootstrap 95% interval {delta_ci[0]:+.5f} to {delta_ci[1]:+.5f})')

fig,ax=plt.subplots(2,2,figsize=(14,9))
for family,color in [('raw','k'),('fft','tab:blue'),('savgol','tab:orange'),('combined','tab:green')]:
    q=summary[summary.family==family]
    ax[0,0].scatter(q.charge_bias*100,q.mean_efficiency,label=family,color=color,s=55)
    ax[0,1].scatter(q.peak_bias*100,q.mean_efficiency,label=family,color=color,s=55)
    ax[1,0].scatter(q.max_fpr_upper95,q.mean_efficiency,label=family,color=color,s=55)
ax[0,0].set(xlabel='absolute charge distortion [%]',ylabel='mean detection efficiency',title='Efficiency versus charge preservation')
ax[0,1].set(xlabel='absolute peak distortion [%]',ylabel='mean detection efficiency',title='Efficiency versus height preservation')
ax[1,0].set(xscale='log',xlabel='largest held-out FPR upper bound',ylabel='mean detection efficiency',title='Efficiency versus false-positive control')

plot_target=1e-3
for family,color in [('raw','k'),('fft','tab:blue'),('savgol','tab:orange'),('combined','tab:green')]:
    q=results[(results.family==family)&(results.target_fpr==plot_target)]
    curve=q.groupby('amplitude').efficiency.max()
    ax[1,1].plot(curve.index,curve.values,'o-',label=f'best {family}',color=color)
ax[1,1].set(xlabel='injected template amplitude',ylabel='held-out efficiency',title='Best family member at 0.1% target FPR');ax[1,1].legend()
for a in ax.flat:a.grid(alpha=.25)
for a in ax[:,:-1].flat:a.legend()
fig.tight_layout();plt.show()

family_wins=choices.family.value_counts() if len(choices) else pd.Series(dtype=int)
print('Conclusion:')
if winner_name!='raw' and delta_ci[0]>0 and len(family_wins) and family_wins.iloc[0]>=2:
    winner=family_wins.index[0]
    print(f'{winner} is the most stable family across the declared distortion tolerances.')
    print(f'The preferred tested setting is {winner_name}; use it only within the validated acquisition bandwidth.')
else:
    print('No filter shows a resolved, parameter-robust improvement over raw reconstruction.')
    print('Retain raw waveforms as the primary result and report mild smoothing only as a cross-check.')
print('This conclusion applies to the measured 08_09 bandwidth, sampling, baseline, and pulse population.')

## Interpretation limits

- The pre-trigger interval is the best available measured null, but it can still contain real dark photoelectrons. Counting one as a false positive is conservative for electronic-noise rejection.
- The injected template represents the normal-trigger dark-pulse morphology in this run. A much faster optical pulse or a changed electronics bandwidth requires repeating the validation.
- A brick-wall FFT filter is nonlocal and can ring around sharp features. Savitzky–Golay is local but can suppress narrow peaks when its window becomes too wide. Their combination inherits both failure modes and is justified only if its held-out improvement exceeds either component alone.
- The final analysis should retain raw waveforms. Filtering is a reconstruction choice that should be reproducible, versioned, and validated again if acquisition settings change.
